# Face Recognition — Model Evaluation

This notebook benchmarks the two recognition engines that power the attendance system:

| Engine | Detector | Recognizer | Era |
|---|---|---|---|
| **Classical** | Haar Cascade | LBPH | 2001 / 2006 |
| **Deep Learning** | YuNet (CNN) | SFace (CNN embeddings) | 2023 / 2021 |

Rather than *claiming* an accuracy number, we **measure** one with a per-identity train/test split, then report precision / recall / F1 and a confusion matrix — the same `evaluate.py` logic that ships with the project, here made interactive.

> Run the cells top-to-bottom. You need a populated `dataset/` (register a few students in the app first). The deep engine additionally needs `python download_models.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # import the project package from notebooks/

import numpy as np
import matplotlib.pyplot as plt

from core import dataset as ds
from evaluate import stratified_split, _eval_classical, _eval_deep, _confusion_png
from core.recognizers import SFaceRecognizer

samples = ds.list_samples(os.path.join('..', 'dataset'))
print(f'Loaded {len(samples)} labelled images across '
      f'{len({l for _, l in samples})} identities')

## 1 — Dataset overview

How many images do we have per student? Class imbalance directly affects recognition quality, so we visualise it first.

In [ ]:
from collections import Counter

if not samples:
    print('No data yet — register students in the app, then re-run this notebook.')
else:
    counts = Counter(label for _, label in samples)
    ids = sorted(counts)
    plt.figure(figsize=(7, 4))
    plt.bar([str(i) for i in ids], [counts[i] for i in ids], color='#1f6feb')
    plt.xlabel('Student ID'); plt.ylabel('# images'); plt.title('Images per identity')
    plt.tight_layout(); plt.show()

## 2 — Evaluate the classical engine (Haar + LBPH)

We hold out 30% of each student's images for testing, train LBPH on the rest, then score the held-out set.

In [ ]:
if samples and len({l for _, l in samples}) >= 2:
    train, test = stratified_split(samples, test_size=0.3)
    y_true, y_pred = _eval_classical(train, test, augment=False)

    from sklearn.metrics import accuracy_score, classification_report
    print(f'Classical accuracy: {accuracy_score(y_true, y_pred):.1%}\n')
    print(classification_report(y_true, y_pred, zero_division=0))
else:
    print('Need at least 2 students to evaluate.')

## 3 — Confusion matrix

Where do the mistakes happen? The diagonal is correct predictions; off-diagonal cells are confusions between specific students.

In [ ]:
if samples and len({l for _, l in samples}) >= 2:
    labels = sorted({l for _, l in samples})
    idx = {c: i for i, c in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)
    for t, p in zip(y_true, y_pred):
        if p in idx:
            cm[idx[t], idx[p]] += 1
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap='Blues')
    plt.xticks(range(len(labels)), labels, rotation=45)
    plt.yticks(range(len(labels)), labels)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix — classical')
    for i in range(len(labels)):
        for j in range(len(labels)):
            plt.text(j, i, cm[i, j], ha='center', va='center')
    plt.colorbar(); plt.tight_layout(); plt.show()

## 4 — Compare against the deep-learning engine (YuNet + SFace)

If the model weights have been downloaded, we run the same split through SFace embeddings + cosine similarity and compare accuracy head-to-head.

In [ ]:
if samples and len({l for _, l in samples}) >= 2 and SFaceRecognizer.is_available():
    yt, yp = _eval_deep(train, test, augment=False)
    from sklearn.metrics import accuracy_score
    print(f'Classical (LBPH)  accuracy: {accuracy_score(y_true, y_pred):.1%}')
    print(f'Deep      (SFace) accuracy: {accuracy_score(yt, yp):.1%}')
else:
    print('Deep engine unavailable — run  python download_models.py  to enable it.')

## Takeaways

- **LBPH** is astonishingly cheap and trains instantly, but its texture-histogram representation degrades under pose and lighting shift, and it does not scale gracefully past a few hundred identities.
- **SFace embeddings** generalise far better because the CNN was trained on millions of faces with a margin loss; cosine similarity in that learned space separates identities cleanly — at the cost of a one-time 37 MB model download.
- The pluggable architecture means the *same* application can ship the lightweight engine by default and upgrade to the deep engine wherever accuracy matters most, with no code changes.